# Pooled-sample subtyping on nuclear expression: does the nucleus carry the same subtypes?

`4_pooled_sample_corrected.ipynb` subtypes the pooled tumor cells of the five Xenium 5K samples (BC, OC, CC, LC,
Prostate; ~624k cells) on the depth-corrected **cytoplasmic** expression of the 634 cytoplasm-enriched target genes
(per-sample depth regression and scaling, pooled PCA / kNN / UMAP) and found cross-sample substates (S-phase, G2/M,
Interferon, Hypoxia, partly ER stress). This notebook repeats **exactly that procedure on the nuclear expression of
the same genes** (`4_pooled_embedding.py --correction sample --compartment nuclear`: the nuclear compartment matrix
with the total nuclear transcripts as size factor, everything else identical; clustering, annotation and figures
from `subtype_utils.py` (one folder up, in `code/1_exploration/`) as before) and asks:

1. **Existence.** Do the cytoplasmic pooled subtypes reappear when the cells are clustered on nuclear expression?
   The per-sample cytoplasmic subtypes of `3_clustering.ipynb` are a secondary reference, compared with the same
   code as in the cytoplasmic pooled notebooks.
2. **Correspondence.** Cell by cell, do the nuclear labels agree with the cytoplasmic pooled labels, do the nuclear
   program scores track the cytoplasmic ones, and are the cytoplasmic subtypes as compact in the nuclear kNN graph
   as in the cytoplasmic graph (neighbourhood label purity)?
3. **Batch effect.** Do the five samples mix in the nuclear embedding?

Expectation: the target genes are cytoplasm-enriched and the nucleus holds fewer transcripts per cell, so the
nuclear embedding should resolve fewer and less separated substates, and the two label sets should correspond only
partly at the cell level.

**Inputs**
- `output/1_exploration/pooled_sample_corrected_nuclear/adata_pooled_nuclear_embedded.h5ad`: all tumor cells x gene
  set on the nuclear compartment, built on HGCC (per sample: normalize_total on the full panel with the nuclear
  depth, gene subset, log1p in `layers["lognorm"]`, depth regression and scaling within the sample; pooled PCA, kNN
  graph, UMAP, t-SNE).
- `output/1_exploration/pooled_sample_corrected/annotation/cell_subtypes_pooled.csv` and the `obs` / kNN graph of
  `pooled_sample_corrected/adata_pooled_cytoplasmic_embedded.h5ad`: the cytoplasmic pooled clusters, labels,
  program scores and 30-NN graph of `4_pooled_sample_corrected.ipynb`, joined on `sample` + `cell_id`.
- `output/1_exploration/<sample>/characteristics/cell_annotations_cytoplasmic.csv` and
  `annotation/cell_subtypes_cytoplasmic.csv`: the per-sample labels and per-cell characteristics of
  `3_clustering.ipynb` (spatial context, cytoplasmic ssGSEA and 8-TF hypoxia, morphology: cell characteristics
  independent of this embedding).
- `data/<sample>/intermediate_data/adata.h5ad` (obs only, via h5py) for the tissue-map backgrounds; `data/_utils/*.gmt`.

**Outputs** (`output/1_exploration/pooled_sample_corrected_nuclear/`; the subfolders are emptied on every run, the
`.h5ad` in the root is kept): the folders of the cytoplasmic pooled notebooks (`clustering/`, `DE/`, `annotation/`,
`spatial/`, `characteristics/`, `comparison/`, tables named `*_nuclear.csv`), plus in `comparison/` the nuclear vs
cytoplasmic files `*_vs_pooled_cyto*`: contingency tables and heatmaps, ARI / NMI, label agreement, program-score
correlations, neighbourhood label purity on both graphs, and the side-by-side tissue maps.

**Naming note.** The shared functions keep their obs column names: in this notebook `subtype_louvain_cyto`,
`cyto_depth` and `cyto_target_counts` hold the **nuclear** cluster, the nuclear depth (total nuclear transcripts on
the panel) and the nuclear target-gene counts; `subtype_label` is the nuclear pooled label and
`subtype_label_pooled_cyto` the cytoplasmic pooled label.

**Saved results for later stages**: `annotation/subtype_annotation_nuclear.csv`, `annotation/cell_subtypes_pooled.csv`
(nuclear, cytoplasmic pooled and per-sample labels per cell) and the nuclear `.h5ad` rewritten with all labels,
program scores and canvas coordinates in `obs`.

**How the nuclear clustering is reviewed**: as for the cytoplasmic pooled object: run with the sweep's suggestion and
no manual labels, review `clustering/resolution_sweep_nuclear.csv`, `annotation/subtype_annotation_nuclear.csv`,
`clustering/cluster_stats_nuclear.csv`, `clustering/cluster_sample_composition.csv`, the `DE/` tables and the
composition of every nuclear cluster in cytoplasmic pooled labels, record the decisions in
`POOLED_CONFIGS[("sample", "nuclear")]` (dated comment) and run again.

## 1. Setup

Imports, paths and the pooled configuration. Parameters, program dictionary, label vocabulary and all analysis
functions come from `subtype_utils.py`, shared with `3_clustering.ipynb`. The six output subfolders are emptied and
recreated on every run.

In [1]:
import datetime
import os
import shutil

import h5py
from anndata.io import read_elem
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# parameters, colours, program dictionary, label vocabulary and the analysis functions shared with 3_clustering.ipynb
import sys
sys.path.insert(0, "..")   # the shared subtype_utils.py is one folder up
import subtype_utils
from subtype_utils import *

set_figsize((8, 8))

/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `annda

In [2]:
utils_path = "../../../data/_utils/"

# CORRECTION selects the pooled object and the output folder, both built by 4_pooled_embedding.py --correction <mode>:
#   "none"    uncorrected pool: depth regression and scaling on the pooled matrix   -> output/1_exploration/pooled/
#   "sample"  depth regression and scaling within each sample, then pooled PCA / kNN -> output/1_exploration/pooled_sample_corrected/
CORRECTION = "sample"
# COMPARTMENT selects the compartment matrix the pooled object was built on ("cytoplasmic", or "nuclear" for the nuclear
# counterpart, 4_pooled_embedding.py --compartment nuclear, whose folder carries the suffix _nuclear)
COMPARTMENT = "nuclear"
POOLED_PATHS = {"none": "../../../output/1_exploration/pooled/", "sample": "../../../output/1_exploration/pooled_sample_corrected/"}
pooled_path = POOLED_PATHS[CORRECTION].rstrip("/") + ("" if COMPARTMENT == "cytoplasmic" else "_" + COMPARTMENT) + "/"
POOLED_OBJECT = f"adata_pooled_{COMPARTMENT}_embedded.h5ad"
NOTEBOOK = "4_pooled" + ("" if CORRECTION == "none" else "_sample_corrected") + ("" if COMPARTMENT == "cytoplasmic" else "_" + COMPARTMENT) + ".ipynb"
subtype_utils.COMPARTMENT = COMPARTMENT   # tag of the shared functions' output file names (*_<compartment>.csv) and figure titles
cyto_pooled_path = POOLED_PATHS["sample"]  # cytoplasmic sample-corrected pooled results: the reference of the nuclear notebook
sample_output = lambda sample: f"../../../output/1_exploration/{sample}/"

paths = {"output": pooled_path}
for name in ["clustering", "DE", "annotation", "spatial", "characteristics", "comparison"]:
    paths[name] = pooled_path + name + "/"
    shutil.rmtree(paths[name], ignore_errors=True)
    os.makedirs(paths[name], exist_ok=True)

In [3]:
SAMPLES = ["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC", "Xenium_5K_LC", "Xenium_5K_Prostate"]
short = lambda sample: sample.replace("Xenium_5K_", "")

# Tissue-map canvas: the samples' coordinates are shifted so that the five tissues sit side by side on one canvas,
# three in the top row and two in the bottom row, with a gap of CANVAS_GAP x the row width between tissues.
CANVAS_LAYOUT = [["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC"], ["Xenium_5K_LC", "Xenium_5K_Prostate"]]
CANVAS_GAP = 0.03
CANVAS_WIDTH_IN = 18   # figure width of the tiled tissue maps; the height follows the canvas aspect ratio

# Pooled clustering configuration per (correction mode, compartment), reviewed like a sample's entry in SAMPLE_CONFIG (3_clustering.ipynb):
# - res_cyto: Louvain resolution (the sweep in section 4 prints a suggestion; this is the reviewed value)
# - manual_labels: reviewed label decisions (pooled cluster id -> label), applied after the automatic annotation
#
# "none" (uncorrected pool), reviewed 2026-11-03 at resolution 1.0 (16 clusters; the sweep suggests 1.2). The pooled clustering follows the samples:
# every cluster except 9 (Interferon), 12 (Myeloid border) and the Low depth clusters is >= 85 % one sample, and each
# sample's bulk falls into one or two clusters (OC 0 + 6, BC 1 + 11, Prostate 2 + 10, CC 3 / 4 / 8, LC 7). Higher
# resolutions (1.2, 1.5) only subdivide within samples (add G2/M, min cluster 32 at 1.2), so 1.0 is kept.
# Label decisions, with the per-sample conventions of SAMPLE_CONFIG:
# - 0 -> Baseline: OC bulk (99 % OC, 66 % of OC cells); markers H19, SULF1, IMPDH2, LAPTM4B, EEF1G are the OC identity
#   genes of the per-sample Baseline clusters; "Oxidative stress" came from the score criterion only (z 0.63), the
#   Stress response overlap is not significant (p 0.02).
# - 1 -> Baseline: BC bulk (98 % BC, 87 % of BC cells); XBP1, TSPAN13, CCND1, FOXA1, RARA are luminal identity genes
#   (BC Baseline clusters 0 / 1 per sample); "ER stress" by score only (z 0.77), Apoptosis overlap p 0.04.
# - 2 -> Baseline (confirmed): Prostate bulk (98 %); NKX3-1, SORD, FOXA1, SPON2 identity, no significant overlap.
# - 7 -> Baseline: the whole LC sample (99.6 % LC, 92 % of LC cells) with 413 markers of lung identity (EPDR1, HOPX,
#   SLC40A1, CDH1, EPAS1); the Autophagy / Lysosome call is the long-marker-list effect noted for LC per sample.
# - 8 -> Stress response: CC cluster with p53 overlap (CDKN1A, GADD45A, SAMD9, BCL2L1, TGFA) and TNFA/NFKB score
#   (z 1.0); 77 % of its cells were Stress response per sample (p53 / TNFA/NFKB -> Stress response convention).
# Kept as annotated: 3 Hypoxia (LDHA, NDRG1, SLC2A1), 4 and 6 S-phase (6 merges the OC S-phase and G2/M cells),
# 9 Interferon (CXCL10, CXCL9, GBP1, STAT1; the only cluster shared by samples), 10 Stroma border (TGFB3, CCN2,
# SFRP1, DKK3 with median fibroblast distance 16 um; these Prostate cells were Stress response per sample),
# 11 Complement / acute phase (C4A, C4B, SERPINA3, CFB), 12 Myeloid border (CD68, ITGB2, SIRPA), 13 Stroma border
# (POSTN, COL5A1, THY1), 5 / 14 / 15 Low depth.
#
# "sample" (per-sample transform, pooled PCA), reviewed 2026-11-03 at resolution 1.0 (16 clusters; the sweep suggests 1.5).
# The samples now mix: nine of the eleven large clusters hold every sample close to its expected share (top-sample
# ratio 1.0-1.9; the Prostate-enriched clusters 8 and 10 are at 3.0 and 2.5), the sample R2 of PC1-5 is 0, and only the
# LC clusters 11 / 12 and the tiny Low depth clusters 13 / 15 are sample-specific (ratio > 3; 14 is 46 CC cells).
# 1.0 is kept: 1.2 gives the same programs with 19 clusters, 1.5 splits to 25 (adds Cholesterol
# synthesis), 0.5 / 0.8 merge G2/M or ER stress into their neighbours. Label decisions (per-sample conventions):
# - 0 -> Baseline (confirmed): 122k cells from all samples, weak identity markers only (EEF1G, IMPDH2, NKX3-1, FOXA1,
#   XBP1, logFC <= 1.5), 69 % per-sample Baseline, no significant overlap (MYC targets p 0.02).
# - 3 -> ER stress: ER chaperones lead the markers (PDIA3, PDIA4, HYOU1, NUCB2, CKAP4) together with the SREBP targets
#   DHCR24, HMGCR, SQLE, EBP; the automatic call was Cholesterol synthesis (5-gene overlap) with ER stress score z 0.50.
#   The per-sample ER stress clusters of BC (46 %), LC (36 %) and Prostate (66 %) land here, as does the Prostate
#   Cholesterol synthesis cluster (35 %); same decision as Prostate cluster 7 per sample (DHCR24 / PDIA3 / PDIA4 ->
#   ER stress), one secretory / ER-lipid axis.
# - 4 -> G2/M: CDC20, NEK2, PLK1, TOP2A, CCNB1, BUB1, CENPF (G2/M score z 1.44); 72 % of its cells were G2/M per
#   sample; the S-phase overlap wins the automatic call only because the S-phase gene list is longer.
# - 5 -> Stress response: p53 overlap (CDKN1A, GADD45A, SAMD9, BCL2L1, TGFA, KLF4) and TNFA/NFKB score, 55 % per-sample
#   Stress response (p53 / TNFA/NFKB -> Stress response convention); 68 % CC, where the Stress response cluster is largest.
# - 9 -> Myeloid border: SIRPA, ITGB2, AXL, LIPA, CD68, CXCR4 are macrophage genes (Myeloid border score z 1.60); the
#   Lysosome overlap is the macrophage-lysosome spill-over noted per sample.
# - 10 -> Low depth (confirmed): median 17 target counts (override), 38 % per-sample Low depth, weak identity markers.
# - 11 -> Hypoxia: 82 % LC (the deep sample, median depth 743), 373 markers led by BLVRA, NQO1, PLS3, HOPX, NDRG1, PRR13,
#   CD99, i.e. the marker list of the LC clusters reviewed as Hypoxia per sample (74 % of its cells were Hypoxia per
#   sample); the Lysosome call is the long-marker-list effect of LC. Sample-specific: the per-sample scaling equalises
#   gene means but not LC's larger within-sample variance.
# - 12 -> Baseline (confirmed): 355 LC cells with neuronal-like markers (GABRB3, PEG10, MAP1B, TCF4) = LC cluster 14,
#   reviewed Baseline per sample (99 % Baseline).
# Kept as annotated: 1 S-phase (PCNA, MYBL2, MCM2-4), 2 Hypoxia (NDRG1, SLC2A1, LDHA, P4HA1), 6 Stroma border (COL5A1,
# POSTN, CTHRC1; fibroblast distance 20 um), 7 Interferon (CXCL10, IFIT1-3, STAT1), 8 Complement / acute phase (CFB,
# SERPINA3, EPAS1; 45 % Prostate, mixes the Prostate Stress response and Complement clusters), 13-15 Low depth.
#
# ("sample", "nuclear") (per-sample transform on the nuclear compartment), reviewed 2026-11-03 after the first pass at
# resolution 1.0 (21 clusters, 11 program labels; the sweep suggests 1.5 with 31 clusters; the same resolution as the cytoplasmic run, so
# the two clusterings are compared at equal granularity). Review of the sweep, annotation, cluster statistics, sample
# composition, DE tables and the composition of every nuclear cluster in cytoplasmic pooled and per-sample labels:
# - 0 -> Baseline (confirmed): 92k cells, the deepest nuclear cluster (median 262 nuclear transcripts), 190 weak markers
#   led by identity genes (XBP1, HSPA8, FOXA1, SEPTIN9, DDR1, SORD, CDH1), ER stress overlap p = 0.016 only and all score
#   z within 0.3; its cytoplasmic labels are the global mix (26 % Baseline, 15 % ER stress, 14 % S-phase).
# - 4 -> G2/M: TPX2, BIRC5, TOP2A, CCNB1, CDC20, PLK1, FOXM1 (G2/M score z 1.57 vs S-phase 1.13; the S-phase overlap is
#   the shared proliferation genes), 41 % cytoplasmic G2/M, 37 % per-sample G2/M; the nuclear counterpart of the
#   cytoplasmic cluster 4 decision.
# - 11 -> Stress response: 75 % Prostate; ZFAND5, EPAS1, GADD45A, NFE2L2, ANXA2 with stromal genes (PALLD, DKK3, SFRP1,
#   CCN2, MMP14) at 17 um from fibroblasts; TNFA/NFKB overlap (-> Stress response convention), Stress response z 0.85,
#   66 % per-sample Stress response (the Prostate Stress response cluster, which the cytoplasmic pool had merged into
#   its Complement / acute phase cluster: 60 % of these cells carry that cytoplasmic label).
# - 14 -> Baseline: a one-gene cluster (IGFBP2 in 80 % of its cells, log2FC 6.2; the next markers MXRA7, TWSG1 are weak);
#   the EMT/stroma call comes from the score criterion alone (IGFBP2 is a hallmark EMT gene), no program overlap
#   (p = 0.13), 42 % per-sample Baseline, fibroblasts 36 um away.
# - 15 -> Myeloid border: CXCR4, LIPA, AXL, GRN, MAN2B1, IL13RA1 are macrophage genes (Myeloid border score z 0.94); the
#   Lysosome overlap is the macrophage-lysosome spill-over; 12 % cytoplasmic Myeloid border, the largest share of any
#   nuclear cluster.
# - 16 -> Hypoxia: 83 % LC (median nuclear depth 747), 323 markers led by CDH1, PLS3, NDRG1, BLVRA, PRR13, HOPX, CD99,
#   i.e. the marker list of the cytoplasmic cluster 11 reviewed as Hypoxia; 70 % cytoplasmic Hypoxia, 69 % per-sample
#   Hypoxia (Hypoxia z 0.81); the Lysosome / Autophagy calls are the long-marker-list effect of the deep sample.
# - 19 -> Stress response: CCND1, TSPAN13, CDKN1A, ANXA2 (TNFA/NFKB score z 0.85, Apoptosis 0.71; TNFA/NFKB -> Stress
#   response convention); 71 % CC, 3.4k cells.
# Kept as annotated: 1 Baseline (EEF1G, IMPDH2), 2 Hypoxia (NDRG1, SLC2A1, LDHA, P4HA1, ALDOA), 3 S-phase (PCNA, MYBL2,
# MCM4, RRM2, TK1, UHRF1), 5 Interferon (IFIT3, STAT1, CXCL10, GBP1, IFIT1), 6 EMT/stroma (COL5A1, MMP14, SEMA3C, ITGA6,
# score z 0.80; fibroblasts at 26.6 um, just outside the 25 um Stroma border override, 23 % cytoplasmic Stroma border),
# 7 ER stress (PDIA4, PDIA3, HYOU1, NUCB2, XBP1), 8 Cholesterol synthesis (DHCR24, SQLE, HMGCR, FDFT1, EBP; z 1.65, a
# program without a cytoplasmic pooled cluster), 10 Complement / acute phase (CFB, SERPINA3), 9 Low depth (median 15
# nuclear target counts) and 12, 13, 17, 18, 20 Low depth: one cluster per sample of cells with no nuclear transcript on
# the panel at all (CC 8878, Prostate 7905, BC 4696, OC 4428, LC 1588 cells).
POOLED_CONFIGS = {
    ("none", "cytoplasmic"): {"res_cyto": 1.0,
                              "manual_labels": {"0": "Baseline", "1": "Baseline", "2": "Baseline", "7": "Baseline",
                                                "8": "Stress response"}},
    ("sample", "cytoplasmic"): {"res_cyto": 1.0,
                                "manual_labels": {"0": "Baseline", "3": "ER stress", "4": "G2/M", "5": "Stress response",
                                                  "9": "Myeloid border", "10": "Low depth", "11": "Hypoxia", "12": "Baseline"}},
    ("sample", "nuclear"): {"res_cyto": 1.0,
                            "manual_labels": {"0": "Baseline", "4": "G2/M", "11": "Stress response", "14": "Baseline",
                                              "15": "Myeloid border", "16": "Hypoxia", "19": "Stress response"}},
}
POOLED_CONFIG = POOLED_CONFIGS[(CORRECTION, COMPARTMENT)]

## 2. Load the pooled object, the per-sample results and the tissue canvas

The pooled object carries, per cell, the sample and the metadata kept by `4_pooled_embedding.py`. The per-sample
results of `3_clustering.ipynb` are joined on `sample` + `cell_id`: the per-sample cluster and label
(`subtype_louvain_sample`, `subtype_label_sample`, `confidence_sample`) and the per-cell characteristics that depend
only on the sample (distances / proximities to the reference cell groups, neighbourhood composition, ssGSEA and 8-TF
hypoxia scores, morphology). They are identical to what a recomputation per sample would give, so they are taken from
the tables rather than recomputed; the program scores, depth statistics, clusters and labels are recomputed on the
pooled object. Every pooled cell must be found in the per-sample tables.

For the tissue maps the samples' coordinates are shifted onto one canvas (`canvas_x`, `canvas_y`): BC, OC, CC in
the top row, LC, Prostate in the bottom row (`CANVAS_LAYOUT`), written to `comparison/canvas_layout.csv`. The
backgrounds of the per-label maps (immune cells, T cells) come from the obs of all cells of each sample.

In [4]:
adata = sc.read_h5ad(pooled_path + POOLED_OBJECT)
adata.obs["sample"] = pd.Categorical(adata.obs["sample"].astype(str), categories=SAMPLES)
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
adata.obs.drop(columns=[c for c in adata.obs.columns if c.startswith("louvain_") or c.startswith("subtype_") or c.startswith("score_")],
               inplace=True, errors="ignore")   # columns of a previous run of this notebook (the .h5ad is rewritten in section 8)
print(f"{adata.shape[0]} tumor cells x {adata.shape[1]} genes; per sample: {adata.obs['sample'].value_counts().loc[SAMPLES].to_dict()}")
print(f"embedding: {adata.uns['embedding_params']}")
assert "X_umap" in adata.obsm and "distances" in adata.obsp, "the pooled object must carry the kNN graph and the UMAP"

# per-sample results of 3_clustering.ipynb, joined on sample + cell_id
spatial_cols = [f"dist_{name}" for name in spatial_refs] + [f"proximity_{name}" for name in spatial_refs] + \
               [f"frac_immune_{RADIUS}um", f"frac_tcell_{RADIUS}um", f"n_cells_{RADIUS}um"]
char_cols = ["hypoxia_ssgsea_cyto", "hypoxia_score_tf", "interferon_ssgsea_cyto", "nucleus_cell_area_ratio"]
per_sample = []
for s in SAMPLES:
    chars = pd.read_csv(sample_output(s) + "characteristics/cell_annotations_cytoplasmic.csv", dtype={"cell_id": str})
    subtypes = pd.read_csv(sample_output(s) + "annotation/cell_subtypes_cytoplasmic.csv", dtype={"cell_id": str})
    assert (chars["cell_id"].values == subtypes["cell_id"].values).all()
    df = chars[["cell_id"] + spatial_cols + char_cols].copy()
    df["subtype_louvain_sample"] = subtypes["subtype_louvain_cyto"].astype(str).values
    df["subtype_label_sample"] = subtypes["subtype_label"].values
    df["confidence_sample"] = subtypes["confidence"].values
    df["resolution_sample"] = subtypes["resolution"].values
    df.index = s + ":" + df["cell_id"].values
    per_sample.append(df.drop(columns="cell_id"))
per_sample = pd.concat(per_sample)
missing = adata.obs_names.difference(per_sample.index)
assert len(missing) == 0, f"{len(missing)} pooled cells are missing from the per-sample tables"
per_sample = per_sample.loc[adata.obs_names]
for col in per_sample.columns:
    adata.obs[col] = per_sample[col].values
sample_labels = [label for label in LABEL_ORDER if label in set(adata.obs["subtype_label_sample"])]
adata.obs["subtype_label_sample"] = pd.Categorical(adata.obs["subtype_label_sample"], categories=sample_labels)
print(f"per-sample labels joined: {len(sample_labels)} labels; "
      + ", ".join(f"{short(s)} {adata.obs.loc[adata.obs['sample'] == s, 'subtype_label_sample'].nunique()}" for s in SAMPLES))
del per_sample

# cytoplasmic sample-corrected pooled results (4_pooled_sample_corrected.ipynb), the reference of this notebook: the
# per-cell labels from the saved record, the per-cell program scores from the saved characteristics table and the
# 30-NN graph from the cytoplasmic object. Only tables and the graph are used, not the obs of the rewritten object,
# so that the notebook also runs on the object as built on HGCC (the kNN graph is the same in both versions).
cyto = pd.read_csv(cyto_pooled_path + "annotation/cell_subtypes_pooled.csv", dtype={"cell_id": str})
cyto.index = cyto["sample"].values + ":" + cyto["cell_id"].values
missing = adata.obs_names.difference(cyto.index)
assert len(missing) == 0, f"{len(missing)} nuclear cells are missing from the cytoplasmic pooled record"
cyto = cyto.loc[adata.obs_names]
adata.obs["subtype_louvain_pooled_cyto"] = cyto["subtype_louvain_pooled"].astype(str).values
cyto_labels = [label for label in LABEL_ORDER if label in set(cyto["subtype_label_pooled"])]
adata.obs["subtype_label_pooled_cyto"] = pd.Categorical(cyto["subtype_label_pooled"].values, categories=cyto_labels)
del cyto
cyto_cells = pd.read_csv(cyto_pooled_path + "characteristics/cell_annotations_cytoplasmic.csv", dtype={"cell_id": str},
                         usecols=lambda c: c in ("sample", "cell_id", "subtype_label") or c.startswith("score_"))
cyto_cells.index = cyto_cells["sample"].values + ":" + cyto_cells["cell_id"].values
assert len(adata.obs_names.difference(cyto_cells.index)) == 0, "nuclear cells missing from the cytoplasmic characteristics table"
cyto_cells = cyto_cells.loc[adata.obs_names]
assert (cyto_cells["subtype_label"].astype(str).values == adata.obs["subtype_label_pooled_cyto"].astype(str).values).all(), \
    "labels of the cytoplasmic record and characteristics table differ"
# program scores keyed by the stored name ("_" for "/": G2_M, EMT_stroma, TNFA_NFKB, Complement_acute phase), as the
# rewritten objects store them
cyto_scores = cyto_cells[[c for c in cyto_cells.columns if c.startswith("score_")]].copy()
cyto_scores.columns = [c[len("score_"):].replace(" / ", "_").replace("/", "_") for c in cyto_scores.columns]
del cyto_cells
with h5py.File(cyto_pooled_path + "adata_pooled_cytoplasmic_embedded.h5ad", "r") as f:
    cyto_names = read_elem(f["obs"]).index
    D_cyto = read_elem(f["obsp"]["distances"]).tocsr()
assert (cyto_names == adata.obs_names).all(), "the cytoplasmic and the nuclear pooled objects must hold the same cells in the same order"
del cyto_names
print(f"cytoplasmic pooled labels joined: {len(cyto_labels)} labels, {cyto_scores.shape[1]} program scores, "
      f"kNN graph with {D_cyto.nnz / D_cyto.shape[0]:.0f} neighbours per cell")


# all cells of every sample (obs only): cell types and coordinates for the tissue-map backgrounds
obs_all = []
for s in SAMPLES:
    with h5py.File(f"../../../data/{s}/intermediate_data/adata.h5ad", "r") as f:
        group = f["obs"]["cell_type_merged"]
        cell_types = group["categories"][:].astype(str)[group["codes"][:]]
        df = pd.DataFrame({"sample": s, "cell_type_merged": cell_types,
                           "global_x": f["obs"]["global_x"][:], "global_y": f["obs"]["global_y"][:]})
    obs_all.append(df)
obs_all = pd.concat(obs_all, ignore_index=True)
print(f"{obs_all.shape[0]} cells of all types in the five samples")

# canvas: shift every sample to its own tile (rows of CANVAS_LAYOUT, top row first)
extent = obs_all.groupby("sample")[["global_x", "global_y"]].agg(["min", "max"])
layout = pd.DataFrame({"x_min": extent[("global_x", "min")], "y_min": extent[("global_y", "min")],
                       "width": extent[("global_x", "max")] - extent[("global_x", "min")],
                       "height": extent[("global_y", "max")] - extent[("global_y", "min")]}).loc[SAMPLES]
row_width = max(layout.loc[row, "width"].sum() for row in CANVAS_LAYOUT)
gap = CANVAS_GAP * row_width
row_heights = [layout.loc[row, "height"].max() for row in CANVAS_LAYOUT]
y_off = 0.0
for row, row_height in zip(reversed(CANVAS_LAYOUT), reversed(row_heights)):   # bottom row at y = 0
    x_off = 0.0
    for s in row:
        layout.loc[s, "x_offset"], layout.loc[s, "y_offset"] = x_off, y_off
        layout.loc[s, "row"] = CANVAS_LAYOUT.index(row)
        x_off += layout.loc[s, "width"] + gap
    y_off += row_height + gap
canvas_w = max(layout.loc[row, "width"].sum() + gap * (len(row) - 1) for row in CANVAS_LAYOUT)
canvas_h = sum(row_heights) + gap * (len(CANVAS_LAYOUT) - 1)
fig_size = (CANVAS_WIDTH_IN, CANVAS_WIDTH_IN * canvas_h / canvas_w)
layout.round(1).to_csv(paths["comparison"] + "canvas_layout.csv")

def to_canvas(df):
    s = df["sample"].astype(str).values
    df["canvas_x"] = df["global_x"].values - layout.loc[s, "x_min"].values + layout.loc[s, "x_offset"].values
    df["canvas_y"] = df["global_y"].values - layout.loc[s, "y_min"].values + layout.loc[s, "y_offset"].values

to_canvas(obs_all)
to_canvas(adata.obs)

def label_canvas(ax):
    """Sample names at the top-left corner of each tile."""
    for s in SAMPLES:
        ax.text(layout.loc[s, "x_offset"], layout.loc[s, "y_offset"] + layout.loc[s, "height"], short(s),
                ha="left", va="bottom", fontsize=11, fontweight="bold")

print(f"canvas {canvas_w:.0f} x {canvas_h:.0f} um, tissue maps at {fig_size[0]:.0f} x {fig_size[1]:.1f} in")
layout.round(0)

623838 tumor cells x 634 genes; per sample: {'Xenium_5K_BC': 102180, 'Xenium_5K_OC': 160250, 'Xenium_5K_CC': 221355, 'Xenium_5K_LC': 44624, 'Xenium_5K_Prostate': 95429}
embedding: {'compartment': 'nuclear', 'depth_correction': 'per-sample OLS residuals on log10(compartment_total_counts + 1) + scale(max_value=10)', 'gene_set_file': 'overlap_genes_3plus_samples.npy', 'n_cells_per_sample': {'Xenium_5K_BC': 102180, 'Xenium_5K_CC': 221355, 'Xenium_5K_LC': 44624, 'Xenium_5K_OC': 160250, 'Xenium_5K_Prostate': 95429}, 'n_comps_pca': 50, 'n_genes': 634, 'n_neighbors': 30, 'n_pcs': 30, 'normalization': "per sample: normalize_total(1e4) on full panel before gene subset, then log1p (layers['lognorm'])", 'sample_correction': 'per-sample depth regression and scaling before pooling (gene means and depth slopes removed within each sample), then pooled PCA / kNN / UMAP / t-SNE', 'samples': array(['Xenium_5K_BC', 'Xenium_5K_OC', 'Xenium_5K_CC', 'Xenium_5K_LC',
       'Xenium_5K_Prostate'], dtype=object)

per-sample labels joined: 15 labels; BC 10, OC 8, CC 11, LC 10, Prostate 9


cytoplasmic pooled labels joined: 11 labels, 23 program scores, kNN graph with 29 neighbours per cell


1790452 cells of all types in the five samples


canvas 34846 x 24296 um, tissue maps at 18 x 12.6 in


,x_min,y_min,width,height,x_offset,y_offset,row
sample,,,,,,,
Xenium_5K_BC,261.0,6.0,9934.0,15504.0,0.0,8792.0,0.0
Xenium_5K_OC,13.0,22.0,11468.0,7955.0,10920.0,8792.0,0.0
Xenium_5K_CC,6.0,646.0,11471.0,10197.0,23375.0,8792.0,0.0
Xenium_5K_LC,9.0,107.0,11482.0,7806.0,0.0,0.0,1.0
Xenium_5K_Prostate,8.0,28.0,11498.0,6420.0,12469.0,0.0,1.0


## 3. Program scores (`build_programs`, `score_programs`)

The same program dictionary as per sample, restricted to the gene set; every cell is scored for every program on its
log-normalized nuclear expression and the scores are z-scored over all pooled cells (so a z of 0.5 means the same
thing in every sample).

In [5]:
programs_raw = load_programs_raw(utils_path)
programs, label_programs = build_programs(set(adata.var_names), programs_raw)
scores_z = score_programs(adata, adata, programs)

S-phase                       47 genes: ANP32E, BIRC5, CBX5, CCNB2, CCNE1, CDC20, CDKN1A, CDKN2A, CDKN3, CHEK1, ...
G2/M                          40 genes: BIRC5, BUB1, CCNA2, CCNB2, CCND1, CDC20, CDKN3, CENPF, CHEK1, CKS2, ...
Interferon                    15 genes: IFIT1, IFIT2, IFIT3, IFI35, IFIH1, STAT1, IFITM1, RSAD2, EIF2AK2, GBP1, ...
Complement / acute phase       7 genes: SERPINA3, CFB, C4A, C4B, UBD, CFH, CD59
Stress response               10 genes: HSPA8, HSPB8, DNAJA1, BAG3, SQSTM1, KLF4, CDKN1A, GADD45A, ZFAND5, SERPINH1
Lysosome                      10 genes: CTSL, LGMN, LIPA, GRN, CREG1, GLB1, GLA, MAN2B1, LAPTM4B, GNS
Cholesterol synthesis          6 genes: HMGCR, SQLE, DHCR24, FDFT1, ACLY, EBP
Hypoxia                       26 genes: ALDOA, ANXA2, CCN2, CDKN1A, CDKN1C, COL5A1, CXCR4, FBP1, HK2, LDHA, ...
Glycolysis                    22 genes: ALDOA, CD44, COL5A1, CXCR4, FUT8, GFPT1, GOT1, HK2, HMMR, IDH1, ...
EMT/stroma                    22 genes: CCN2, CD44, CD59, CD

## 4. Resolution sweep and clustering (`resolution_sweep`, `cluster`)

Louvain on the pooled kNN graph (30 PCs, 30 neighbours, built on HGCC) at the resolutions of `RES_GRID`; the sweep
reports how many programs the score criterion resolves at each value and suggests the coarsest resolution that
resolves all of them. The clustering uses `POOLED_CONFIG["res_cyto"]`, the reviewed value.

In [6]:
resolution_sweep(adata, scores_z, label_programs, POOLED_CONFIG, paths)
cluster(adata, adata, POOLED_CONFIG["res_cyto"])

 resolution  n_clusters  min_cluster_size  n_program_labels                                                                                                                                                                           programs
        0.5          12               343                 5                                                                                                                                EMT/stroma, Hypoxia, Interferon, S-phase, TNFA/NFKB
        0.8          18                71                 9                                                          Cholesterol synthesis, Complement / acute phase, EMT/stroma, G2/M, Hypoxia, Interferon, Lysosome, Myeloid border, S-phase
        1.0          21              1588                11                                    Cholesterol synthesis, Complement / acute phase, EMT/stroma, ER stress, G2/M, Hypoxia, Interferon, Lysosome, Myeloid border, S-phase, TNFA/NFKB
        1.2          24               152   

Number of clusters (nuclear, resolution 1.0): 21


## 5. Cluster statistics and sample composition (`cluster_statistics`)

Depth check of the pooled embedding (Spearman of PC1-5 with log depth), per-cluster depth, target-gene counts, area
and distances (the inputs of the annotation overrides), and, specific to the pooled analysis, the **sample
composition of every cluster**: the share of each sample among the cluster's cells next to the sample's share of all
cells. `top_sample_ratio` = share of the dominant sample / its expected share; a cluster with a ratio near
1 / (expected share) consists of one sample only and is a candidate sample-specific state (or a batch artifact).

In [7]:
cluster_stats = cluster_statistics(adata, adata, paths)

expected = adata.obs["sample"].value_counts(normalize=True).loc[SAMPLES]
composition_cl = pd.crosstab(adata.obs["subtype_louvain_cyto"], adata.obs["sample"], normalize="index")[SAMPLES]
composition_cl["n_cells"] = adata.obs["subtype_louvain_cyto"].value_counts()
composition_cl["top_sample"] = composition_cl[SAMPLES].idxmax(axis=1)
composition_cl["top_share"] = composition_cl[SAMPLES].max(axis=1)
composition_cl["top_sample_ratio"] = composition_cl["top_share"] / expected.loc[composition_cl["top_sample"]].values
composition_cl.round(3).to_csv(paths["clustering"] + "cluster_sample_composition.csv")
print("expected shares: " + ", ".join(f"{short(s)} {v:.2f}" for s, v in expected.items()))
composition_cl.round(2)

Spearman(PC1, log10 nuclear depth) = +0.22
Spearman(PC2, log10 nuclear depth) = -0.16


Spearman(PC3, log10 nuclear depth) = -0.06
Spearman(PC4, log10 nuclear depth) = +0.02


Spearman(PC5, log10 nuclear depth) = -0.04
                      n_cells  median_cyto_depth  median_cyto_target_counts  median_cell_area  median_dist_immune  median_dist_fibroblast  low_depth
subtype_louvain_cyto                                                                                                                                
0                       92338              262.0                       66.0              97.9                35.9                    36.7      False
1                       89106              121.0                       27.0              68.5                35.8                    30.7      False
2                       66912              124.0                       34.0              76.3                51.1                    56.1      False
3                       60038              178.0                       47.0              83.2                41.9                    39.0      False
4                       59924              189.0               

sample,Xenium_5K_BC,Xenium_5K_OC,Xenium_5K_CC,Xenium_5K_LC,Xenium_5K_Prostate,n_cells,top_sample,top_share,top_sample_ratio
subtype_louvain_cyto,,,,,,,,,
0,0.20,0.21,0.28,0.12,0.19,92338,Xenium_5K_CC,0.28,0.79
1,0.17,0.38,0.22,0.06,0.16,89106,Xenium_5K_OC,0.38,1.48
2,0.13,0.31,0.41,0.04,0.10,66912,Xenium_5K_CC,0.41,1.16
3,0.17,0.30,0.41,0.04,0.08,60038,Xenium_5K_CC,0.41,1.17
4,0.14,0.37,0.40,0.03,0.06,59924,Xenium_5K_CC,0.40,1.12
5,0.14,0.18,0.52,0.07,0.09,51879,Xenium_5K_CC,0.52,1.47
6,0.14,0.20,0.51,0.05,0.10,49741,Xenium_5K_CC,0.51,1.45
7,0.18,0.19,0.31,0.10,0.21,29448,Xenium_5K_CC,0.31,0.88
8,0.17,0.19,0.39,0.04,0.20,28282,Xenium_5K_CC,0.39,1.10


## 6. Differential expression (`differential_expression`)

Wilcoxon test of every pooled cluster against all other pooled cells on `layers["lognorm"]`, same thresholds as per
sample; one table per cluster in `DE/`.

In [8]:
de_dict = differential_expression(adata, paths)

Cluster 0: 190 marker genes; top: XBP1, HSPA8, SEPTIN9, FOXA1, DAP, TSPAN13, CSNK1A1, IRAK1
Cluster 1: 2 marker genes; top: EEF1G, IMPDH2
Cluster 2: 16 marker genes; top: NDRG1, SLC2A1, LDHA, APOL1, P4HA1, ALDOA, H19, ANXA2
Cluster 3: 71 marker genes; top: PCNA, MYBL2, MCM4, TUBB, RRM2, DHFR, PRKDC, DNAJC9
Cluster 4: 100 marker genes; top: TPX2, BIRC5, TOP2A, SMC4, KPNA2, CENPF, TUBB, CCNB1
Cluster 5: 20 marker genes; top: IFIT3, MALL, STAT1, CDKN1A, SAMD9, BCL2L1, HOPX, CXCL10
Cluster 6: 17 marker genes; top: CDKN1A, COL5A1, APP, MMP14, SEMA3C, ITGA6, MYH9, MSN
Cluster 7: 15 marker genes; top: PDIA4, PDIA3, HYOU1, NUCB2, GRN, XBP1, LGMN, CDH1
Cluster 8: 13 marker genes; top: DHCR24, SQLE, HMGCR, FDFT1, EBP, EMP2, CDH1, PAK1IP1
Cluster 9: 1 marker genes; top: SQSTM1
Cluster 10: 24 marker genes; top: CFB, SERPINA3, CD47, SDC4, BACE2, TNIP1, DDR1, XBP1
Cluster 11: 83 marker genes; top: ZFAND5, EEF1G, EPAS1, ANXA2, GADD45A, PALLD, SOX9, NFE2L2
Cluster 12: 0 marker genes; top: 
Cluster 13:

## 7. Annotation (`annotate`)

The per-sample rule (marker overlap, program score, Baseline, Low depth and Stroma border overrides, confidence) on
the pooled clusters, followed by the reviewed decisions of `POOLED_CONFIG["manual_labels"]`. The per-cell record
`annotation/cell_subtypes_pooled.csv` lists, for every cell, the pooled cluster and label next to the per-sample
cluster and label.

In [9]:
annotation, categories = annotate(adata, adata, de_dict, cluster_stats, scores_z, programs, label_programs, POOLED_CONFIG, paths)

cell_subtypes = pd.DataFrame({"sample": adata.obs["sample"].astype(str).values, "cell_id": adata.obs["cell_id"].values,
                              "subtype_louvain_pooled": adata.obs["subtype_louvain_cyto"].astype(str).values,
                              "subtype_label_pooled": adata.obs["subtype_label"].astype(str).values})
cell_subtypes["confidence_pooled"] = cell_subtypes["subtype_louvain_pooled"].map(annotation["confidence"].to_dict())
cell_subtypes["resolution_pooled"] = POOLED_CONFIG["res_cyto"]
for col in ["subtype_louvain_pooled_cyto", "subtype_label_pooled_cyto", "subtype_louvain_sample", "subtype_label_sample", "confidence_sample", "resolution_sample"]:
    cell_subtypes[col] = adata.obs[col].astype(str).values if col.startswith("subtype") else adata.obs[col].values
cell_subtypes.to_csv(paths["annotation"] + "cell_subtypes_pooled.csv", index=False)

21 clusters -> 12 labels; clusters to review: none
         n_cells                     label confidence  n_markers           overlap_program  overlap_genes     overlap_p             score_program  score_z
cluster                                                                                                                                                   
0          92338                  Baseline   reviewed        190                 ER stress             11  1.553629e-02                 ER stress     0.26
1          89106                  Baseline       high          2               MYC targets              1  7.433433e-02                 Autophagy     0.09
2          66912                   Hypoxia       high         16                   Hypoxia              6  1.585449e-05                   Hypoxia     1.02
3          60038                   S-phase       high         71                   S-phase             31  0.000000e+00                   S-phase     1.22
4          59924   

## 8. Subtype characteristics (`summarize_characteristics`) and saved objects

Per-label medians and IQRs of the characteristics (program scores recomputed on the pooled object; spatial context,
ssGSEA / 8-TF hypoxia and morphology from the per-sample tables) and the per-cell table, as per sample. The pooled
`.h5ad` is then rewritten with the clusters, labels (pooled and per-sample), program scores and canvas coordinates in
`obs` and the configuration in `uns["subtyping"]`, so that later stages can load one object with everything. In the
saved object the "/" of four program-score column names is replaced by "_" (`score_G2_M`, `score_EMT_stroma`,
`score_TNFA_NFKB`, `score_Complement_acute phase`), because h5py reads "/" as a group separator.

In [10]:
summarize_characteristics(adata, paths, extra_cell_cols=["sample", "subtype_louvain_pooled_cyto", "subtype_label_pooled_cyto", "subtype_louvain_sample", "subtype_label_sample"])

# subtype_louvain (set by cluster()) and subtype_louvain_cyto are the same column on the pooled object: keep one in the file
adata.obs.drop(columns=[f"louvain_{res}" for res in RES_GRID] + ["subtype_louvain_cyto"], inplace=True, errors="ignore")
adata.uns["subtyping"] = {"date": datetime.date.today().isoformat(), "res_cyto": POOLED_CONFIG["res_cyto"],
                          "manual_labels": {k: v for k, v in POOLED_CONFIG["manual_labels"].items()},
                          "labels": list(categories), "notebook": NOTEBOOK, "compartment": COMPARTMENT, "correction": CORRECTION}
# h5py treats "/" in a column name as a group separator, so the program-score columns "score_G2/M", "score_EMT/stroma",
# "score_TNFA/NFKB" and "score_Complement / acute phase" are stored with "_" instead (the label names themselves are
# categorical values and unaffected); the in-memory object keeps the renamed columns from here on
adata.obs.columns = [c.replace(" / ", "_").replace("/", "_") for c in adata.obs.columns]
# write to a temporary file first, so that a failed write cannot leave a truncated object behind
adata.write(pooled_path + POOLED_OBJECT + ".tmp")
os.replace(pooled_path + POOLED_OBJECT + ".tmp", pooled_path + POOLED_OBJECT)
adata.obs["subtype_louvain_cyto"] = adata.obs["subtype_louvain"]   # name expected by the shared functions
print(f"pooled object rewritten with {len(categories)} labels: {list(categories)}")

                          n_cells  score_Hypoxia  hypoxia_ssgsea_cyto  hypoxia_score_tf  score_Interferon  interferon_ssgsea_cyto  score_Complement / acute phase  score_S-phase  score_G2/M  score_Stress response  score_Lysosome  proximity_immune  proximity_tcell  proximity_cd8_tcell  proximity_myeloid  proximity_bcell  proximity_fibroblast  proximity_endothelial  frac_immune_30um  frac_tcell_30um  n_cells_30um  cyto_depth  cyto_target_counts  cell_area  nucleus_cell_area_ratio
subtype_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
S-phase                     60038   

pooled object rewritten with 12 labels: ['S-phase', 'G2/M', 'Interferon', 'Complement / acute phase', 'Stress response', 'Cholesterol synthesis', 'Hypoxia', 'EMT/stroma', 'ER stress', 'Baseline', 'Myeloid border', 'Low depth']


## 9. Figures (`make_figures` + pooled-specific figures)

The per-sample figure set (UMAP by cluster and by label, PCA loadings, tissue map by label, composition, program
heatmap, marker dotplot, one map per label on the other tumor cells / immune cells / T cells, box plots, maps of
scores and proximities), with the tissue maps drawn on the tiled canvas. In addition, in `clustering/`:
`umap_sample.png` and `tsne_sample.png` (cells coloured by sample, drawn in random order so that no sample hides
another) with one panel per sample, and the stacked sample composition of every cluster and of every label.

In [11]:
make_figures(obs_all, adata, adata, de_dict, annotation, categories, scores_z, "Pooled", paths, fig_size, xy=("canvas_x", "canvas_y"))

sample_colors = {s: clr.to_hex(plt.get_cmap("tab10")(i)) for i, s in enumerate(SAMPLES)}
cell_colors = np.array([sample_colors[s] for s in adata.obs["sample"].astype(str)])
rng = np.random.default_rng(0)
order = rng.permutation(adata.shape[0])
for key, name in [("X_umap", "umap"), ("X_tsne", "tsne")]:
    if key not in adata.obsm:
        continue
    emb = adata.obsm[key]
    fig, axes = plt.subplots(1, len(SAMPLES) + 1, figsize=(5 * (len(SAMPLES) + 1), 5))
    axes[0].scatter(emb[order, 0], emb[order, 1], c=cell_colors[order], s=0.3, linewidths=0, rasterized=True)
    axes[0].set_title(f"{name.upper()}: all samples")
    for ax, s in zip(axes[1:], SAMPLES):
        is_s = (adata.obs["sample"] == s).values
        ax.scatter(emb[~is_s, 0], emb[~is_s, 1], c="lightgray", s=0.3, linewidths=0, rasterized=True)
        ax.scatter(emb[is_s, 0], emb[is_s, 1], c=[sample_colors[s]], s=0.3, linewidths=0, rasterized=True)
        ax.set_title(f"{short(s)} ({is_s.sum()} cells)")
    for ax in axes:
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(paths["clustering"] + f"{name}_sample.png", dpi=200, bbox_inches="tight")
    plt.close()

# stacked sample composition per cluster and per label
for groupby, fname in [("subtype_louvain_cyto", "cluster_sample_composition"), ("subtype_label", "label_sample_composition")]:
    comp = pd.crosstab(adata.obs[groupby], adata.obs["sample"], normalize="index")[SAMPLES]
    if groupby == "subtype_label":
        comp = comp.loc[categories]
    fig, ax = plt.subplots(figsize=(0.5 * len(comp) + 3, 4.5))
    bottom = np.zeros(len(comp))
    for s in SAMPLES:
        ax.bar(comp.index.astype(str), comp[s].values, bottom=bottom, color=sample_colors[s], label=short(s))
        bottom += comp[s].values
    ax.axhline(1, color="none")
    ax.grid(False)
    ax.set_ylabel("share of the cluster's cells")
    ax.set_xticklabels(comp.index.astype(str), rotation=60, ha="right")
    ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
    ax.set_title("expected: " + ", ".join(f"{short(s)} {v:.2f}" for s, v in expected.items()), fontsize=9)
    plt.savefig(paths["clustering"] + f"{fname}.png", dpi=200, bbox_inches="tight")
    plt.close()

## 10. Comparison with the per-sample subtypes (`comparison/`)

**(1) Existence.** `label_presence.csv`: every label that occurs in any analysis, with its share of the pooled cells
and of each sample's cells under the per-sample labels, and whether the pooled clustering has it. Pooled clusters
whose dominant sample exceeds three times its expected share (`cluster_sample_composition.csv`) are listed as
sample-specific.

**(2) Correspondence.** Per sample, the contingency table of per-sample label x pooled label, row-normalised
(`correspondence_<sample>.csv`, heatmaps); the adjusted Rand index and normalised mutual information between the
two partitions of the sample's cells, at label level and at cluster level (`correspondence_scores.csv`); and the
**label agreement** (`label_agreement.csv`): for every per-sample label of every sample, the share of its cells that
carry the same pooled label, next to the best-matching pooled label and its share (which catches a program that the
pooled annotation names differently, e.g. TNFA/NFKB vs Stress response). Tiled tissue maps of the per-sample labels
and the pooled labels side by side make the agreement visible in space.

**(3) Batch mixing.** On the pooled kNN graph (the 30 nearest neighbours of every cell in PC space) the share of a
cell's neighbours that come from its own sample is compared with the sample's share of all cells:
`same_sample_ratio` = observed / expected (1 = perfectly mixed; 1 / expected share = the cell's neighbourhood is
its own sample only), together with the normalised entropy of the neighbourhood's sample composition (1 = all five
samples in proportion, 0 = one sample). Medians per sample and per pooled label go to `batch_mixing.csv`; the
one-way-ANOVA R2 of sample on each of PC1-10 (`pc_sample_variance.csv`) says how much of each component is a sample
effect. The UMAP / t-SNE panels of section 9 are the visual check. In the `none` version these statistics measure
the raw sample effect; in the `sample` version the gene means are equal across samples by construction, so they
measure whether residual within-sample structure (covariance, cell-state-specific shifts) still separates the samples.

In [12]:
comp_path = paths["comparison"]
obs = adata.obs

# (1) existence of the per-sample labels in the pooled clustering
presence = pd.DataFrame(index=[label for label in LABEL_ORDER if label in set(sample_labels) | set(categories)])
presence["pooled_pct"] = (obs["subtype_label"].value_counts(normalize=True) * 100).reindex(presence.index).fillna(0)
for s in SAMPLES:
    presence[f"{short(s)}_pct"] = (obs.loc[obs["sample"] == s, "subtype_label_sample"].value_counts(normalize=True) * 100).reindex(presence.index).fillna(0)
presence["n_samples"] = (presence[[f"{short(s)}_pct" for s in SAMPLES]] > 0).sum(axis=1)
presence["in_pooled"] = presence["pooled_pct"] > 0
presence["pooled_clusters"] = [", ".join(annotation.index[annotation["label"] == label]) for label in presence.index]
presence.round(2).to_csv(comp_path + "label_presence.csv")
sample_specific = composition_cl[composition_cl["top_sample_ratio"] > 3].copy()
sample_specific["label"] = annotation.loc[sample_specific.index, "label"]
sample_specific.round(3).to_csv(comp_path + "sample_specific_clusters.csv")
print("per-sample labels missing from the pooled clustering: " + (", ".join(presence.index[(presence["n_samples"] > 0) & ~presence["in_pooled"]]) or "none"))
print("pooled labels absent per sample: " + (", ".join(presence.index[(presence["n_samples"] == 0) & presence["in_pooled"]]) or "none"))
print(f"sample-specific pooled clusters (top sample > 3x expected): "
      + (", ".join(f"{cl} ({r.label}, {short(r.top_sample)} {r.top_share:.0%})" for cl, r in sample_specific.iterrows()) or "none"))

# (2) correspondence per sample
scores, agreement, tables = [], [], {}
for s in SAMPLES:
    o = obs[obs["sample"] == s]
    table = pd.crosstab(o["subtype_label_sample"], o["subtype_label"], normalize="index")
    table = table.reindex(index=[l for l in LABEL_ORDER if l in table.index], columns=[l for l in LABEL_ORDER if l in table.columns])
    table.round(4).to_csv(comp_path + f"correspondence_{s}.csv")
    tables[s] = table
    scores.append({"sample": s, "n_cells": len(o),
                   "ARI_label": adjusted_rand_score(o["subtype_label_sample"].astype(str), o["subtype_label"].astype(str)),
                   "NMI_label": normalized_mutual_info_score(o["subtype_label_sample"].astype(str), o["subtype_label"].astype(str)),
                   "ARI_cluster": adjusted_rand_score(o["subtype_louvain_sample"].astype(str), o["subtype_louvain_cyto"].astype(str)),
                   "NMI_cluster": normalized_mutual_info_score(o["subtype_louvain_sample"].astype(str), o["subtype_louvain_cyto"].astype(str))})
    for label in table.index:
        agreement.append({"sample": s, "label": label, "n_cells": int((o["subtype_label_sample"] == label).sum()),
                          "agreement": table.loc[label, label] if label in table.columns else 0.0,
                          "best_pooled_label": table.loc[label].idxmax(), "best_share": table.loc[label].max()})
scores = pd.DataFrame(scores).set_index("sample")
scores.round(3).to_csv(comp_path + "correspondence_scores.csv")
agreement = pd.DataFrame(agreement)
agreement.round(4).to_csv(comp_path + "label_agreement.csv", index=False)
print(scores.round(3).to_string())

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(5.5 * len(SAMPLES), 6))
for ax, s in zip(axes, SAMPLES):
    sns.heatmap(tables[s], cmap="Blues", vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 6}, cbar=False, ax=ax)
    ax.grid(False)
    ax.set_title(f"{short(s)}: ARI {scores.loc[s, 'ARI_label']:.2f}, NMI {scores.loc[s, 'NMI_label']:.2f}")
    ax.set_xlabel("pooled label")
    ax.set_ylabel("per-sample label" if s == SAMPLES[0] else "")
plt.tight_layout()
plt.savefig(comp_path + "correspondence_heatmaps.png", dpi=200, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(0.8 * len(presence) + 2, 4.5))
for i, s in enumerate(SAMPLES):
    d = agreement[agreement["sample"] == s].set_index("label").reindex(presence.index)
    ax.plot(np.arange(len(presence)), d["agreement"].values, "o", color=sample_colors[s], label=short(s))
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.grid(False)
ax.set_xticks(np.arange(len(presence)))
ax.set_xticklabels(presence.index, rotation=60, ha="right")
ax.set_ylabel("share of cells with the same pooled label")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False)
plt.savefig(comp_path + "label_agreement.png", dpi=200, bbox_inches="tight")
plt.close()

# per-sample labels vs pooled labels on the canvas
fig, axes = plt.subplots(1, 2, figsize=(2 * fig_size[0], fig_size[1]))
for ax, col, title in zip(axes, ["subtype_label_sample", "subtype_label"], ["per-sample labels", "pooled labels"]):
    ax.scatter(obs["canvas_x"], obs["canvas_y"], c=obs[col].astype(str).map(LABEL_COLORS).values, s=0.4, linewidths=0, rasterized=True)
    label_canvas(ax)
    ax.grid(False)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=LABEL_COLORS[l], label=l) for l in LABEL_ORDER if l in set(sample_labels) | set(categories)]
axes[1].legend(handles=handles, frameon=False, bbox_to_anchor=(1, 1), loc="upper left", fontsize=9)
plt.savefig(comp_path + "scatter_labels_sample_vs_pooled.png", dpi=200, bbox_inches="tight")
plt.close()

# (3) batch mixing on the pooled kNN graph
D = adata.obsp["distances"].tocsr()
codes = obs["sample"].cat.codes.values
rows = np.repeat(np.arange(D.shape[0]), np.diff(D.indptr))
n_neighbors = np.diff(D.indptr)
same = (codes[rows] == codes[D.indices]).astype(float)
# cells without a stored neighbour (identical expression vectors, i.e. zero counts: their zero-distance neighbours are
# not kept in the sparse graph) have no neighbourhood: NaN, so that they are left out of the medians below
has_nn = n_neighbors > 0
print(f"batch mixing: {(~has_nn).sum()} cells without kNN neighbours excluded")
same_frac = np.where(has_nn, np.bincount(rows, weights=same, minlength=D.shape[0]) / np.maximum(n_neighbors, 1), np.nan)
counts = sparse.csr_matrix((np.ones(len(rows)), (rows, codes[D.indices])), shape=(D.shape[0], len(SAMPLES))).toarray()
p = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
with np.errstate(divide="ignore", invalid="ignore"):
    entropy = np.clip(-np.nansum(np.where(p > 0, p * np.log(p), 0), axis=1) / np.log(len(SAMPLES)), 0, 1)
entropy = np.where(has_nn, entropy, np.nan)
obs["same_sample_frac"] = same_frac
obs["same_sample_ratio"] = same_frac / expected.loc[obs["sample"].astype(str)].values
obs["neighbourhood_entropy"] = entropy
mixing = pd.concat([obs.groupby("sample", observed=True)[["same_sample_frac", "same_sample_ratio", "neighbourhood_entropy"]].median().assign(group="sample"),
                    obs.groupby("subtype_label", observed=True)[["same_sample_frac", "same_sample_ratio", "neighbourhood_entropy"]].median().assign(group="label")])
mixing["expected_frac"] = [expected.get(i, np.nan) for i in mixing.index]
mixing["max_ratio"] = [1 / expected[i] if i in expected.index else np.nan for i in mixing.index]
mixing.round(3).to_csv(comp_path + "batch_mixing.csv")
print(mixing.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.boxplot(data=obs, x="sample", y="same_sample_ratio", order=SAMPLES, palette=sample_colors, showfliers=False, ax=axes[0])
axes[0].axhline(1, color="gray", lw=0.8, ls="--")
axes[0].set_title("same-sample neighbours / expected (1 = mixed)")
axes[0].set_xticklabels([short(s) for s in SAMPLES])
sns.boxplot(data=obs, x="sample", y="neighbourhood_entropy", order=SAMPLES, palette=sample_colors, showfliers=False, ax=axes[1])
axes[1].set_title("neighbourhood sample entropy (1 = all samples in proportion)")
axes[1].set_xticklabels([short(s) for s in SAMPLES])
sns.boxplot(data=obs, x="subtype_label", y="same_sample_ratio", order=categories, palette=LABEL_COLORS, showfliers=False, ax=axes[2])
axes[2].axhline(1, color="gray", lw=0.8, ls="--")
axes[2].set_title("same-sample ratio per pooled label")
axes[2].set_xticklabels(categories, rotation=60, ha="right")
for ax in axes:
    ax.grid(False)
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(comp_path + "batch_mixing.png", dpi=200, bbox_inches="tight")
plt.close()

# variance of the PCs explained by sample (one-way ANOVA R2)
pcs = pd.DataFrame(adata.obsm["X_pca"][:, :10], columns=[f"PC{i + 1}" for i in range(10)], index=obs.index)
pcs["sample"] = obs["sample"].values
total_ss = ((pcs.iloc[:, :10] - pcs.iloc[:, :10].mean()) ** 2).sum()
group_means = pcs.groupby("sample", observed=True).mean()
group_sizes = pcs.groupby("sample", observed=True).size()
between_ss = ((group_means - pcs.iloc[:, :10].mean()) ** 2).mul(group_sizes, axis=0).sum()
pc_var = pd.DataFrame({"variance_ratio": adata.uns["pca"]["variance_ratio"][:10], "sample_R2": (between_ss / total_ss).values}, index=total_ss.index)
pc_var.round(4).to_csv(comp_path + "pc_sample_variance.csv")
print(pc_var.round(3).T.to_string())

per-sample labels missing from the pooled clustering: Lysosome, MYC targets, Stroma border
pooled labels absent per sample: none
sample-specific pooled clusters (top sample > 3x expected): 11 (Stress response, Prostate 75%), 13 (Low depth, Prostate 100%), 16 (Hypoxia, LC 83%), 17 (Low depth, BC 100%), 18 (Low depth, OC 100%), 20 (Low depth, LC 99%)


                    n_cells  ARI_label  NMI_label  ARI_cluster  NMI_cluster
sample                                                                     
Xenium_5K_BC         102180      0.049      0.054        0.032        0.050
Xenium_5K_OC         160250      0.134      0.130        0.110        0.119
Xenium_5K_CC         221355      0.056      0.074        0.061        0.074
Xenium_5K_LC          44624      0.076      0.103        0.099        0.142
Xenium_5K_Prostate    95429      0.047      0.068        0.048        0.071


batch mixing: 28694 cells without kNN neighbours excluded


                          same_sample_frac  same_sample_ratio  neighbourhood_entropy   group  expected_frac  max_ratio
Xenium_5K_BC                         0.241              1.474                  0.738  sample          0.164      6.105
Xenium_5K_OC                         0.276              1.074                  0.724  sample          0.257      3.893
Xenium_5K_CC                         0.621              1.749                  0.626  sample          0.355      2.818
Xenium_5K_LC                         0.138              1.928                  0.806  sample          0.072     13.980
Xenium_5K_Prostate                   0.276              1.803                  0.740  sample          0.153      6.537
S-phase                              0.414              1.477                  0.688   label            NaN        NaN
G2/M                                 0.448              1.555                  0.647   label            NaN        NaN
Interferon                           0.483      

                  PC1    PC2    PC3    PC4    PC5    PC6    PC7    PC8    PC9   PC10
variance_ratio  0.007  0.004  0.003  0.003  0.003  0.003  0.002  0.002  0.002  0.002
sample_R2       0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000


## 11. Nuclear vs cytoplasmic pooled labels (`comparison/*_vs_pooled_cyto*`)

The cell-level comparison that this notebook is for. Reference: the cytoplasmic sample-corrected pooled labels
(`subtype_label_pooled_cyto`); test: the nuclear pooled labels (`subtype_label`).

- **Correspondence.** Contingency table cytoplasmic label x nuclear label, row-normalised, over all cells and per
  sample (`correspondence_vs_pooled_cyto[_<sample>].csv`, heatmaps); ARI / NMI of the two partitions at label and
  cluster level (`correspondence_scores_vs_pooled_cyto.csv`); and the **label agreement**: for every cytoplasmic
  pooled label the share of its cells that carry the same nuclear label, next to the best-matching nuclear label
  (`label_agreement_vs_pooled_cyto.csv`, dot plot).
- **Program scores.** Spearman correlation, over all cells and within each sample, of the nuclear program score
  (section 3, on nuclear `lognorm`) with the cytoplasmic pooled score of the same program
  (`score_correlation_vs_pooled_cyto.csv`, bar plot): whether the nuclear expression carries the program signal at
  all, independently of any clustering.
- **Neighbourhood label purity.** For every cell, the share of its 30 nearest neighbours that carry the cell's own
  cytoplasmic pooled label, on the nuclear kNN graph and on the cytoplasmic kNN graph; mean per label
  (`label_purity_vs_pooled_cyto.csv`, paired bars, with the label's overall share as the purity of a random
  neighbourhood). A ratio nuclear / cytoplasmic well below 1 means the cytoplasmic subtype is dispersed in nuclear
  space: "more mixed" in one number per label.
- **Tissue maps** of the cytoplasmic pooled labels and the nuclear pooled labels side by side
  (`scatter_labels_pooled_cyto_vs_nuclear.png`).

In [13]:
ref_col, nuc_col = "subtype_label_pooled_cyto", "subtype_label"
stored = lambda name: name.replace(" / ", "_").replace("/", "_")   # program name -> stored score column name (cyto_scores)
groups = {"all": obs}
groups.update({s: obs[obs["sample"] == s] for s in SAMPLES})
gname = lambda g: "all cells" if g == "all" else short(g)
fname = lambda g: "" if g == "all" else "_" + g

# correspondence: contingency, ARI / NMI, label agreement (all cells and per sample)
scores_nc, agreement_nc, tables_nc = [], [], {}
for g, o in groups.items():
    table = pd.crosstab(o[ref_col], o[nuc_col], normalize="index")
    table = table.reindex(index=[l for l in LABEL_ORDER if l in table.index], columns=[l for l in LABEL_ORDER if l in table.columns])
    table.round(4).to_csv(comp_path + f"correspondence_vs_pooled_cyto{fname(g)}.csv")
    tables_nc[g] = table
    scores_nc.append({"group": g, "n_cells": len(o),
                      "ARI_label": adjusted_rand_score(o[ref_col].astype(str), o[nuc_col].astype(str)),
                      "NMI_label": normalized_mutual_info_score(o[ref_col].astype(str), o[nuc_col].astype(str)),
                      "ARI_cluster": adjusted_rand_score(o["subtype_louvain_pooled_cyto"].astype(str), o["subtype_louvain_cyto"].astype(str)),
                      "NMI_cluster": normalized_mutual_info_score(o["subtype_louvain_pooled_cyto"].astype(str), o["subtype_louvain_cyto"].astype(str))})
    for label in table.index:
        agreement_nc.append({"group": g, "label": label, "n_cells": int((o[ref_col] == label).sum()),
                             "agreement": table.loc[label, label] if label in table.columns else 0.0,
                             "best_nuclear_label": table.loc[label].idxmax(), "best_share": table.loc[label].max()})
scores_nc = pd.DataFrame(scores_nc).set_index("group")
scores_nc.round(3).to_csv(comp_path + "correspondence_scores_vs_pooled_cyto.csv")
agreement_nc = pd.DataFrame(agreement_nc)
agreement_nc.round(4).to_csv(comp_path + "label_agreement_vs_pooled_cyto.csv", index=False)
print("nuclear vs cytoplasmic pooled labels")
print(scores_nc.round(3).to_string())
print(agreement_nc[agreement_nc["group"] == "all"].drop(columns="group").round(3).to_string(index=False))

fig, axes = plt.subplots(1, len(groups), figsize=(5.5 * len(groups), 6))
for ax, (g, table) in zip(axes, tables_nc.items()):
    sns.heatmap(table, cmap="Blues", vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 6}, cbar=False, ax=ax)
    ax.grid(False)
    ax.set_title(f"{gname(g)}: ARI {scores_nc.loc[g, 'ARI_label']:.2f}, NMI {scores_nc.loc[g, 'NMI_label']:.2f}")
    ax.set_xlabel("nuclear pooled label")
    ax.set_ylabel("cytoplasmic pooled label" if g == "all" else "")
plt.tight_layout()
plt.savefig(comp_path + "correspondence_heatmaps_vs_pooled_cyto.png", dpi=200, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(0.8 * len(cyto_labels) + 2, 4.5))
for g in groups:
    d = agreement_nc[agreement_nc["group"] == g].set_index("label").reindex(cyto_labels)
    ax.plot(np.arange(len(cyto_labels)), d["agreement"].values, "o" if g != "all" else "D",
            color=sample_colors[g] if g != "all" else "black", ms=5 if g != "all" else 7, label=gname(g), zorder=3 if g == "all" else 2)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.grid(False)
ax.set_xticks(np.arange(len(cyto_labels)))
ax.set_xticklabels(cyto_labels, rotation=60, ha="right")
ax.set_ylabel("share of cells with the same nuclear label")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False)
plt.savefig(comp_path + "label_agreement_vs_pooled_cyto.png", dpi=200, bbox_inches="tight")
plt.close()

# program scores: Spearman of the nuclear score with the cytoplasmic pooled score, all cells and per sample
score_corr = []
for name in programs:
    if stored(name) not in cyto_scores.columns:
        continue
    nuc, cyt = scores_z[name].values, cyto_scores[stored(name)].values
    row = {"program": name, "all": spearmanr(nuc, cyt).correlation}
    for s in SAMPLES:
        is_s = (obs["sample"] == s).values
        row[short(s)] = spearmanr(nuc[is_s], cyt[is_s]).correlation
    score_corr.append(row)
score_corr = pd.DataFrame(score_corr).set_index("program")
score_corr.round(3).to_csv(comp_path + "score_correlation_vs_pooled_cyto.csv")
print("Spearman(nuclear program score, cytoplasmic pooled program score):")
print(score_corr.round(2).to_string())

fig, ax = plt.subplots(figsize=(0.6 * len(score_corr) + 2, 4.5))
xs = np.arange(len(score_corr))
ax.bar(xs, score_corr["all"].values, color="lightgray", label="all cells")
for s in SAMPLES:
    ax.plot(xs, score_corr[short(s)].values, "o", color=sample_colors[s], ms=4, label=short(s))
ax.axhline(0, color="black", lw=0.6)
ax.grid(False)
ax.set_xticks(xs)
ax.set_xticklabels(score_corr.index, rotation=60, ha="right")
ax.set_ylabel("Spearman rho, nuclear vs cytoplasmic score")
ax.set_ylim(-0.2, 1)
ax.legend(frameon=False, fontsize=8)
plt.savefig(comp_path + "score_correlation_vs_pooled_cyto.png", dpi=200, bbox_inches="tight")
plt.close()

# neighbourhood label purity of the cytoplasmic pooled labels on the nuclear and on the cytoplasmic kNN graph
def label_purity(D, codes):
    rows = np.repeat(np.arange(D.shape[0]), np.diff(D.indptr))
    same = (codes[rows] == codes[D.indices]).astype(float)
    n = np.diff(D.indptr)   # cells without a stored neighbour (zero counts) get NaN and drop out of the means
    return np.where(n > 0, np.bincount(rows, weights=same, minlength=D.shape[0]) / np.maximum(n, 1), np.nan)

codes_cyto = obs[ref_col].cat.codes.values
obs["purity_cyto_graph"] = label_purity(D_cyto, codes_cyto)
obs["purity_nuclear_graph"] = label_purity(adata.obsp["distances"].tocsr(), codes_cyto)
purity = obs.groupby(ref_col, observed=True)[["purity_cyto_graph", "purity_nuclear_graph"]].mean().loc[cyto_labels]
purity.insert(0, "n_cells", obs[ref_col].value_counts().loc[cyto_labels])
purity["random"] = obs[ref_col].value_counts(normalize=True).loc[cyto_labels]   # purity of a random neighbourhood
purity["ratio_nuclear_cyto"] = purity["purity_nuclear_graph"] / purity["purity_cyto_graph"]
purity.round(4).to_csv(comp_path + "label_purity_vs_pooled_cyto.csv")
print("neighbourhood purity of the cytoplasmic pooled labels (mean share of same-label neighbours):")
print(purity.round(3).to_string())

fig, ax = plt.subplots(figsize=(0.9 * len(purity) + 2, 4.5))
xs = np.arange(len(purity))
ax.bar(xs - 0.2, purity["purity_cyto_graph"].values, width=0.4, color="tab:blue", label="cytoplasmic kNN graph")
ax.bar(xs + 0.2, purity["purity_nuclear_graph"].values, width=0.4, color="tab:orange", label="nuclear kNN graph")
ax.plot(xs, purity["random"].values, "_", color="black", ms=14, mew=1.5, label="random neighbourhood")
ax.grid(False)
ax.set_xticks(xs)
ax.set_xticklabels(purity.index, rotation=60, ha="right")
ax.set_ylabel("share of neighbours with the same cytoplasmic label")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8)
plt.savefig(comp_path + "label_purity_vs_pooled_cyto.png", dpi=200, bbox_inches="tight")
plt.close()
del D_cyto

# cytoplasmic pooled labels vs nuclear pooled labels on the canvas
fig, axes = plt.subplots(1, 2, figsize=(2 * fig_size[0], fig_size[1]))
for ax, col, title in zip(axes, [ref_col, nuc_col], ["cytoplasmic pooled labels", "nuclear pooled labels"]):
    ax.scatter(obs["canvas_x"], obs["canvas_y"], c=obs[col].astype(str).map(LABEL_COLORS).values, s=0.4, linewidths=0, rasterized=True)
    label_canvas(ax)
    ax.grid(False)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=LABEL_COLORS[l], label=l) for l in LABEL_ORDER if l in set(cyto_labels) | set(categories)]
axes[1].legend(handles=handles, frameon=False, bbox_to_anchor=(1, 1), loc="upper left", fontsize=9)
plt.savefig(comp_path + "scatter_labels_pooled_cyto_vs_nuclear.png", dpi=200, bbox_inches="tight")
plt.close()

nuclear vs cytoplasmic pooled labels
                    n_cells  ARI_label  NMI_label  ARI_cluster  NMI_cluster
group                                                                      
all                  623838      0.054      0.073        0.050        0.078
Xenium_5K_BC         102180      0.034      0.044        0.029        0.042
Xenium_5K_OC         160250      0.112      0.114        0.117        0.110
Xenium_5K_CC         221355      0.057      0.071        0.062        0.069
Xenium_5K_LC          44624      0.077      0.094        0.084        0.115
Xenium_5K_Prostate    95429      0.043      0.065        0.038        0.062
                   label  n_cells  agreement best_nuclear_label  best_share
                 S-phase    88290      0.274            S-phase       0.274
                    G2/M    57030      0.435               G2/M       0.435
              Interferon    32990      0.272         Interferon       0.272
Complement / acute phase    22902      0.090    Str

Spearman(nuclear program score, cytoplasmic pooled program score):
                              all    BC    OC    CC    LC  Prostate
program                                                            
S-phase                      0.52  0.31  0.48  0.29  0.32      0.23
G2/M                         0.37  0.21  0.44  0.24  0.29      0.12
Interferon                   0.26  0.18  0.18  0.21  0.40      0.18
Complement / acute phase     0.26  0.22  0.21  0.18  0.25      0.23
Stress response              0.14  0.11  0.09  0.16  0.09      0.14
Lysosome                     0.23  0.06  0.07  0.13  0.26      0.05
Cholesterol synthesis        0.09  0.07  0.08  0.09  0.14      0.08
Hypoxia                      0.43  0.08  0.25  0.33  0.28      0.17
Glycolysis                   0.16  0.05  0.12  0.10  0.15      0.07
EMT/stroma                   0.18  0.12  0.12  0.15  0.24      0.12
ER stress                    0.33  0.12  0.08  0.05  0.23      0.14
Oxidative stress             0.23  0.05  0.11  0.

neighbourhood purity of the cytoplasmic pooled labels (mean share of same-label neighbours):
                           n_cells  purity_cyto_graph  purity_nuclear_graph  random  ratio_nuclear_cyto
subtype_label_pooled_cyto                                                                              
S-phase                      88290              0.668                 0.219   0.142               0.328
G2/M                         57030              0.706                 0.248   0.091               0.351
Interferon                   32990              0.640                 0.138   0.053               0.215
Complement / acute phase     22902              0.582                 0.196   0.037               0.337
Stress response              52847              0.666                 0.269   0.085               0.404
Hypoxia                      88264              0.674                 0.241   0.141               0.357
ER stress                    75564              0.577                 0.135

## 12. Summary

Criteria used for the three answers (`comparison/summary.txt`):

1. A cytoplasmic pooled label counts as **recovered** when the nuclear clustering has it and its label agreement over all cells is at least 0.5 (half of its cells carry the same nuclear label); the per-sample labels are judged as in the cytoplasmic notebooks (median agreement over the samples that have the label). The artifact classes (Baseline, Stroma border, Myeloid border, Low depth) are reported but not counted.
2. **Correspondence** is summarised by the ARI / NMI of nuclear vs cytoplasmic pooled labels (all cells and per sample), the median Spearman correlation of the program scores and the median purity ratio over the cytoplasmic labels, plus the per-sample ARI / NMI at label level.
3. **Mixing** is good when the median same-sample ratio of every sample is below 2 (less than twice the expected share
   of own-sample neighbours) and no PC among the first five has a sample R2 above 0.2.

In [14]:
artifact_labels = {"Baseline", "Stroma border", "Myeloid border", "Low depth"}
n_str = ", ".join(f"{short(s)} {n}" for s, n in adata.obs["sample"].value_counts().loc[SAMPLES].items())
lines = [f"Pooled subtyping (correction: {CORRECTION}, compartment: {COMPARTMENT}) of {adata.shape[0]} tumor cells ({n_str}), "
         f"resolution {POOLED_CONFIG['res_cyto']}, {len(annotation)} clusters -> {len(categories)} labels", ""]

lines.append("(1) Existence of the cytoplasmic pooled subtypes in the nuclear clustering")
a_all = agreement_nc[agreement_nc["group"] == "all"].set_index("label")
cyto_pct = obs[ref_col].value_counts(normalize=True) * 100
recovered_nc, lost_nc = [], []
for label in cyto_labels:
    if label in artifact_labels:
        continue
    per = ", ".join(f"{short(s)} {r.agreement:.2f}" for r in agreement_nc[(agreement_nc["group"] != "all") & (agreement_nc["label"] == label)].itertuples()
                    for s in [r.group])
    ok = label in categories and a_all.loc[label, "agreement"] >= 0.5
    (recovered_nc if ok else lost_nc).append(label)
    lines.append(f"  {label:26s} cytoplasmic {cyto_pct.get(label, 0):.1f}%; nuclear {presence.loc[label, 'pooled_pct'] if label in presence.index else 0:.1f}% "
                 f"({'present' if label in categories else 'absent'}); agreement all {a_all.loc[label, 'agreement']:.2f} ({per}); "
                 f"best nuclear label {a_all.loc[label, 'best_nuclear_label']} {a_all.loc[label, 'best_share']:.2f} -> {'recovered' if ok else 'not recovered'}")
lines.append(f"  recovered: {', '.join(recovered_nc) or 'none'}")
lines.append(f"  not recovered: {', '.join(lost_nc) or 'none'}")
lines.append(f"  nuclear-only labels: {', '.join(l for l in categories if l not in cyto_labels) or 'none'}")
lines.append("")
lines.append("(1b) Existence of the per-sample subtypes in the nuclear clustering (secondary reference)")

recovered, lost = [], []
for label in presence.index:
    if label in artifact_labels or presence.loc[label, "n_samples"] == 0:
        continue
    a = agreement[(agreement["label"] == label)]
    med = a["agreement"].median()
    where = ", ".join(f"{short(r.sample)} {r.agreement:.2f}" for r in a.itertuples())
    ok = presence.loc[label, "in_pooled"] and med >= 0.5
    (recovered if ok else lost).append(label)
    lines.append(f"  {label:26s} per-sample in {int(presence.loc[label, 'n_samples'])} sample(s); pooled {presence.loc[label, 'pooled_pct']:.1f}% "
                 f"({'present' if presence.loc[label, 'in_pooled'] else 'absent'}); agreement {where} -> {'recovered' if ok else 'not recovered'}")
new_pooled = [l for l in categories if presence.loc[l, "n_samples"] == 0]
lines.append(f"  recovered: {', '.join(recovered) or 'none'}")
lines.append(f"  not recovered: {', '.join(lost) or 'none'}")
lines.append(f"  pooled-only labels: {', '.join(new_pooled) or 'none'}")
specific_str = ", ".join(f"{cl} ({annotation.loc[cl, 'label']})" for cl in sample_specific.index) or "none"
lines.append(f"  sample-specific pooled clusters: {specific_str}")
lines.append("")

lines.append("(2) Correspondence of nuclear and cytoplasmic pooled labels, cell by cell")
for g in groups:
    lines.append(f"  {gname(g):9s} ARI {scores_nc.loc[g, 'ARI_label']:.2f}, NMI {scores_nc.loc[g, 'NMI_label']:.2f} (labels); "
                 f"ARI {scores_nc.loc[g, 'ARI_cluster']:.2f}, NMI {scores_nc.loc[g, 'NMI_cluster']:.2f} (clusters)")
lines.append(f"  program scores: median Spearman(nuclear, cytoplasmic) {score_corr['all'].median():.2f} over {len(score_corr)} programs "
             f"(range {score_corr['all'].min():.2f} to {score_corr['all'].max():.2f}; highest "
             + ", ".join(f"{p} {r:.2f}" for p, r in score_corr["all"].sort_values(ascending=False).head(3).items()) + ")")
non_artifact = [l for l in cyto_labels if l not in artifact_labels]
lines.append(f"  neighbourhood purity of the cytoplasmic labels: median ratio nuclear / cytoplasmic graph {purity.loc[non_artifact, 'ratio_nuclear_cyto'].median():.2f} "
             f"(" + ", ".join(f"{l} {purity.loc[l, 'purity_nuclear_graph']:.2f} vs {purity.loc[l, 'purity_cyto_graph']:.2f}" for l in non_artifact) + ")")
lines.append("")
lines.append("(2b) Correspondence of per-sample and nuclear pooled labels within each sample")

for s in SAMPLES:
    lines.append(f"  {short(s):9s} ARI {scores.loc[s, 'ARI_label']:.2f}, NMI {scores.loc[s, 'NMI_label']:.2f} (labels); "
                 f"ARI {scores.loc[s, 'ARI_cluster']:.2f}, NMI {scores.loc[s, 'NMI_cluster']:.2f} (clusters)")
lines.append("")
lines.append("(3) Batch mixing in the pooled embedding")
for s in SAMPLES:
    lines.append(f"  {short(s):9s} median same-sample ratio {mixing.loc[s, 'same_sample_ratio']:.2f} (1 = mixed, {mixing.loc[s, 'max_ratio']:.1f} = separated), "
                 f"neighbourhood entropy {mixing.loc[s, 'neighbourhood_entropy']:.2f}")
lines.append("  sample R2 of PC1-5: " + ", ".join(f"{pc} {r:.2f}" for pc, r in pc_var['sample_R2'].head(5).items()))
mixed = (mixing.loc[SAMPLES, "same_sample_ratio"] < 2).all() and (pc_var["sample_R2"].head(5) < 0.2).all()
lines.append(f"  -> {'no strong batch effect' if mixed else 'sample effect present'} by the criteria above")
summary_text = "\n".join(lines)
open(comp_path + "summary.txt", "w").write(summary_text + "\n")
print(summary_text)

Pooled subtyping (correction: sample, compartment: nuclear) of 623838 tumor cells (BC 102180, OC 160250, CC 221355, LC 44624, Prostate 95429), resolution 1.0, 21 clusters -> 12 labels

(1) Existence of the cytoplasmic pooled subtypes in the nuclear clustering
  S-phase                    cytoplasmic 14.2%; nuclear 9.6% (present); agreement all 0.27 (BC 0.30, OC 0.35, CC 0.25, LC 0.21, Prostate 0.08); best nuclear label S-phase 0.27 -> not recovered
  G2/M                       cytoplasmic 9.1%; nuclear 9.6% (present); agreement all 0.43 (BC 0.41, OC 0.57, CC 0.37, LC 0.27, Prostate 0.10); best nuclear label G2/M 0.43 -> not recovered
  Interferon                 cytoplasmic 5.3%; nuclear 8.3% (present); agreement all 0.27 (BC 0.20, OC 0.30, CC 0.30, LC 0.38, Prostate 0.08); best nuclear label Interferon 0.27 -> not recovered
  Complement / acute phase   cytoplasmic 3.7%; nuclear 2.0% (present); agreement all 0.09 (BC 0.13, OC 0.08, CC 0.04, LC 0.06, Prostate 0.10); best nuclear label S